## Database Connection

In [ ]:
!pip install -q pymongo dnspython plotly pandas python-dotenv fastapi uvicorn nest_asyncio pyngrok httpx requests

import os
import json
from pymongo import MongoClient
from bson import json_util
from dotenv import load_dotenv

load_dotenv()

URI = os.getenv('MONGO_URI', 'mongodb+srv://<username>:<password>@<cluster>.mongodb.net/?appName=<appName>')
DB_NAME = os.getenv('DB_NAME', 'tracera')
COLLECTION_NAME = os.getenv('COLLECTION_NAME', 'raw_linkedin')

def connect_to_mongodb_and_get_data(uri, db_name, collection_name, query={}):
    """Menghubungkan ke MongoDB Atlas dan mengekstrak dokumen dalam format JSON mentah."""
    if not uri or "<username>" in uri:
        print("[!] Peringatan: MONGO_URI belum diset. Menggunakan mock mode jika diperlukan.")
        return []
    try:
        client = MongoClient(uri)
        db = client[db_name]
        collection = db[collection_name]
        documents = list(collection.find(query))
        json_data = json.loads(json_util.dumps(documents))
        print(f"Berhasil terhubung! {len(json_data)} dokumen dimuat dari koleksi '{collection_name}'.")
        return json_data
    except Exception as e:
        print(f"Error koneksi database: {e}")
        return []

raw_documents = connect_to_mongodb_and_get_data(URI, DB_NAME, COLLECTION_NAME)


## Pre-processing Data

In [ ]:
from datetime import datetime, timezone
import pandas as pd

# ==============================================================================
# 1. KAMUS TRANSLASI & NORMALISASI SKILLS (Bilingual & Canonical Normalization)
# ==============================================================================
SKILL_TRANSLATIONS = {
    # a. Hard Skills - Desain, Media & Kreatif Teknis
    'desain': 'Design',
    'design': 'Design',
    'desain grafis': 'Graphic Design',
    'design grafis': 'Graphic Design',
    'graphic design': 'Graphic Design',
    'pemikiran desain': 'Design Thinking',
    'creative direction': 'Creative Direction',
    'creative strategy': 'Creative Strategy',
    'penulisan': 'Writing',
    'writing': 'Writing',
    'creative writing': 'Creative Writing',
    'penulisan kreatif': 'Creative Writing',
    'report writing': 'Report Writing',
    'penulisan laporan': 'Report Writing',
    'technical writing': 'Technical Writing',
    'penulisan teknis': 'Technical Writing',
    'scientific writing': 'Scientific Writing',
    'penulisan ilmiah': 'Scientific Writing',
    'proposal writing': 'Proposal Writing',
    'penulisan proposal': 'Proposal Writing',

    # b. Hard Skills - Data & Analisis
    'analisis data': 'Data Analysis',
    'analisa data': 'Data Analysis',
    'analisis': 'Data Analysis',
    'data analysis': 'Data Analysis',
    'analisis bisnis': 'Business Analysis',
    'business analysis': 'Business Analysis',
    'analisis keuangan': 'Financial Analysis',
    'financial analysis': 'Financial Analysis',
    'analisis sosial': 'Social Analysis',
    'pengolahan data': 'Data Processing',
    'entri data': 'Data Entry',
    'data entry': 'Data Entry',
    'mangalisis suatu proyek': 'Project Analysis',
    'riset': 'Research',
    'penelitian': 'Research',
    'research': 'Research',
    'research skills': 'Research',
    'analytical skills': 'Analytical Skills',
    'keahlian analitis': 'Analytical Skills',
    'analytic problem solving': 'Analytical Skills',

    # c. Hard Skills - Manajemen Proyek, Operasional & IT / Tools
    'manajemen proyek': 'Project Management',
    'project management': 'Project Management',
    'dokumen proyek': 'Project Documentation',
    'manajemen operasional': 'Operations Management',
    'operations management': 'Operations Management',
    'pengembangan web': 'Web Development',
    'manajemen file': 'File Management',
    'sistem informasi geografis (gis)': 'GIS',
    'geographic information systems (gis)': 'GIS',
    'arcgis': 'GIS',
    'sistem manajemen keselamatan dan kesehatan kerja': 'Occupational Health and Safety (OHS)',
    'kesehatan, keselamatan, dan lingkungan (hse)': 'Health, Safety, and Environment (HSE)',
    'peningkatan proses bisnis': 'Business Process Improvement',
    'monitoring dan evaluasi': 'Monitoring & Evaluation',
    'perencanaan strategis': 'Strategic Planning',
    'strategic planning': 'Strategic Planning',
    'manajemen perubahan': 'Change Management',
    'change management': 'Change Management',
    'organizational development': 'Organizational Development',
    'pengembangan organisasi': 'Organizational Development',
    'collaboration solutions': 'Collaboration Tools',
    'python (programming language)': 'Python',
    'go (programming language)': 'Go',
    'golang': 'Go',
    'c (programming language)': 'C',
    'r (programming language)': 'R',
    'java (programming language)': 'Java',
    'powerpoint': 'Microsoft PowerPoint',
    'excel': 'Microsoft Excel',
    'word': 'Microsoft Word',

    # d. Hard Skills - Keuangan, Akuntansi & Hukum Bisnis
    'keuangan': 'Finance',
    'finance': 'Finance',
    'jasa keuangan': 'Financial Services',
    'manajemen keuangan': 'Financial Management',
    'financial management': 'Financial Management',
    'perencanaan keuangan': 'Financial Planning',
    'financial planning': 'Financial Planning',
    'akuntansi': 'Accounting',
    'accounting': 'Accounting',
    'akuntansi keuangan': 'Financial Accounting',
    'financial accounting': 'Financial Accounting',
    'pembukuan': 'Bookkeeping',
    'bookkeeping': 'Bookkeeping',
    'perpajakan': 'Taxation',
    'tax': 'Taxation',
    'taxations': 'Taxation',
    'perencanaan anggaran': 'Budgeting',
    'budgeting': 'Budgeting',
    'budgets': 'Budgeting',
    'budget management': 'Budgeting',
    'auditing': 'Auditing',
    'internal audits': 'Auditing',
    'financial audits': 'Auditing',
    'financial reporting': 'Financial Reporting',
    'financial statements': 'Financial Reporting',
    'mengolah data biaya': 'Cost Accounting',
    'cost accounting': 'Cost Accounting',
    'konsultan manajemen': 'Management Consulting',
    'konsultan lingkungan': 'Environmental Consulting',
    'hukum': 'Legal & Law',
    'negosiasi kontrak': 'Contract Negotiation',
    'contract negotiation': 'Contract Negotiation',

    # e. Hard Skills - Pemasaran, Komunikasi Korporat, Penjualan & SDM
    'pemasaran': 'Marketing',
    'marketing': 'Marketing',
    'pemasaran digital': 'Digital Marketing',
    'pemasaran media sosial': 'Social Media Marketing',
    'social media marketing': 'Social Media Marketing',
    'pemasaran produk': 'Product Marketing',
    'strategi pemasaran': 'Marketing Strategy',
    'marketing strategy': 'Marketing Strategy',
    'strategi konten': 'Content Strategy',
    'content strategy': 'Content Strategy',
    'pengoptimalan media sosial (social media optimization/smo)': 'Social Media Optimization (SMO)',
    'komunikasi merek': 'Brand Communication',
    'brand communication': 'Brand Communication',
    'social media communications': 'Social Media Marketing',
    'hubungan masyarakat': 'Public Relations',
    'public relations': 'Public Relations',
    'penjualan': 'Sales',
    'sales': 'Sales',
    'perencanaan acara': 'Event Planning',
    'event planning': 'Event Planning',
    'event management': 'Event Planning',
    'sumber daya manusia (sdm)': 'Human Resources (HR)',
    'human resources (hr)': 'Human Resources (HR)',
    'human resources': 'Human Resources (HR)',
    'perencanaan sumber daya manusia strategis': 'Strategic Human Resource Planning',
    'administrasi': 'Administration',
    'administration': 'Administration',
    'office administration': 'Administration',
    'administrasi perkantoran': 'Administration',
    'bekerja di shift': 'Shift Work',
    'opname kerja': 'Stock Opname',

    # f. Hard Skills - Bahasa & Pengajaran
    'english': 'English',
    'bahasa inggris': 'English',
    'bahasa indonesia': 'Indonesian',
    'teaching': 'Teaching',
    'pengajaran': 'Teaching',

    # g. Soft Skills - Komunikasi & Interpersonal
    'komunikasi': 'Communication',
    'communication': 'Communication',
    'communications': 'Communication',
    'kemampuan komunikasi': 'Communication',
    'interpersonal communication': 'Communication',
    'written communication': 'Communication',
    'strategic communications': 'Communication',
    'intercultural communication': 'Communication',
    'cross-cultural communication skills': 'Communication',
    'komunikasi interpersonal': 'Communication',
    'komunikasi tertulis': 'Communication',
    'active listening': 'Active Listening',
    'mendengarkan aktif': 'Active Listening',
    'persuasi': 'Persuasion',
    'persuasion': 'Persuasion',
    'layanan pelanggan': 'Customer Service',
    'customer service': 'Customer Service',
    'pelayanan': 'Customer Service',
    'keahlian interpersonal': 'Interpersonal Skills',
    'interpersonal skills': 'Interpersonal Skills',
    'interpersonal': 'Interpersonal Skills',
    'keahlian presentasi': 'Presentation Skills',
    'presentation skills': 'Presentation Skills',
    'presentation': 'Presentation Skills',
    'presentations': 'Presentation Skills',
    'presentasi': 'Presentation Skills',
    'public speaking': 'Public Speaking',
    'berbicara di depan umum': 'Public Speaking',
    'relationship building': 'Relationship Building',
    'membangun relasi': 'Relationship Building',

    # h. Soft Skills - Tim & Kerjasama
    'kerja sama tim': 'Teamwork',
    'kerja tim': 'Teamwork',
    'kerjasama tim': 'Teamwork',
    'kemampuan kerja tim': 'Teamwork',
    'teamwork': 'Teamwork',
    'team work': 'Teamwork',
    'kolaborasi tim': 'Collaboration',
    'collaboration': 'Collaboration',
    'team collaboration': 'Collaboration',
    'cross-team collaboration': 'Collaboration',
    'cross-functional collaborations': 'Collaboration',
    'resolusi konflik': 'Conflict Resolution',
    'conflict resolution': 'Conflict Resolution',
    'conflict management': 'Conflict Resolution',
    'manajemen konflik': 'Conflict Resolution',

    # i. Soft Skills - Kepemimpinan & Manajemen Orang
    'kepemimpinan': 'Leadership',
    'leadership': 'Leadership',
    'team leadership': 'Leadership',
    'kepemimpinan tim': 'Leadership',
    'project leadership': 'Leadership',
    'organizational leadership': 'Leadership',
    'strategic leadership': 'Leadership',
    'cross-functional team leadership': 'Leadership',
    'leadership development': 'Leadership Development',
    'pengembangan kepemimpinan': 'Leadership Development',
    'manajemen tim': 'Team Management',
    'team management': 'Team Management',
    'team coordination': 'Team Management',
    'team building': 'Team Management',
    'team motivation': 'Team Management',
    'people management': 'People Management',
    'manajemen orang': 'People Management',
    'people development': 'People Development',
    'team development': 'People Development',
    'pengembangan tim': 'People Development',
    'coaching': 'Coaching & Mentoring',
    'mentoring': 'Coaching & Mentoring',
    'coaching & mentoring': 'Coaching & Mentoring',
    'training & mentoring': 'Coaching & Mentoring',
    'pelatihan & mentoring': 'Coaching & Mentoring',
    'sales coaching': 'Coaching & Mentoring',
    'kemampuan berorganisasi': 'Organization Skills',
    'organization skills': 'Organization Skills',
    'organizational skills': 'Organization Skills',
    'manajemen': 'Management',
    'management': 'Management',

    # j. Soft Skills - Pemecahan Masalah & Berpikir Kritis
    'pemecahan masalah': 'Problem Solving',
    'memecahkan masalah': 'Problem Solving',
    'problem solving': 'Problem Solving',
    'creative problem solving': 'Problem Solving',
    'collaborative problem solving': 'Problem Solving',
    'team problem solving': 'Problem Solving',
    'berpikir kritis': 'Critical Thinking',
    'critical thinking': 'Critical Thinking',
    'critical thinker': 'Critical Thinking',
    'analytical thinking': 'Critical Thinking',
    'pengambilan keputusan': 'Decision Making',
    'decision making': 'Decision Making',
    'decision-making': 'Decision Making',
    'data-driven decision making': 'Decision Making',
    'strategic thinking': 'Strategic Thinking',
    'creativity': 'Creativity',
    'kreativitas': 'Creativity',
    'creative thinking': 'Creativity',
    'creativity skills': 'Creativity',
    'creativity and innovation': 'Creativity',

    # k. Soft Skills - Manajemen Diri & Sikap Kerja
    'manajemen waktu': 'Time Management',
    'time management': 'Time Management',
    'manajemen diri': 'Self Management',
    'self management': 'Self Management',
    'adaptability': 'Adaptability',
    'adaptable': 'Adaptability',
    'easily adaptable': 'Adaptability',
    'adaptation': 'Adaptability',
    'flexibility': 'Adaptability',
    'negosiasi': 'Negotiation',
    'negotiation': 'Negotiation',
    'negotiation skills': 'Negotiation',
    'emotional intelligence': 'Emotional Intelligence',
    'work ethic': 'Work Ethic',
    'etika kerja': 'Work Ethic',
    'working under pressure': 'Working Under Pressure',
    'bekerja dalam tekanan': 'Working Under Pressure',
    'hard working': 'Work Ethic',
    'bekerja keras': 'Work Ethic',
    'jujur dan disiplin': 'Honesty & Discipline',
    'honesty & discipline': 'Honesty & Discipline',
    'bintal dan jasmani': 'Mental & Physical Discipline',
    'mental & physical discipline': 'Mental & Physical Discipline',
    'organizational culture': 'Organizational Culture'
}

# ==============================================================================
# 2. TAKSONOMI RESMI SOFT SKILLS (Bentuk Kanonikal Hasil Normalisasi)
# ==============================================================================
# Disimpan dalam huruf kecil untuk matching case-insensitive terhadap s_norm
SOFT_SKILLS = {
    # Komunikasi & Interpersonal
    'communication',
    'public speaking',
    'presentation skills',
    'active listening',
    'persuasion',
    'customer service',
    'interpersonal skills',
    'relationship building',

    # Tim, Kerjasama, & Hubungan
    'teamwork',
    'collaboration',
    'conflict resolution',

    # Kepemimpinan & People Management
    'leadership',
    'leadership development',
    'team management',
    'people management',
    'people development',
    'coaching & mentoring',
    'organization skills',
    'management',
    'organizational culture',

    # Pemecahan Masalah & Berpikir Kritis
    'problem solving',
    'critical thinking',
    'decision making',
    'strategic thinking',
    'creativity',

    # Adaptabilitas, Manajemen Waktu, & Sikap Kerja
    'adaptability',
    'time management',
    'self management',
    'negotiation',
    'emotional intelligence',
    'work ethic',
    'working under pressure',
    'honesty & discipline',
    'mental & physical discipline'
}

# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================
def normalize_skill(name):
    """Membersihkan dan menstandardisasi nama keahlian."""
    if not name or not isinstance(name, str):
        return ''
    clean = name.strip()
    return SKILL_TRANSLATIONS.get(clean.lower(), clean)


def clean_job_title(title):
    """Membersihkan nama jabatan dari noise keterangan atau nama institusi."""
    if not title or not isinstance(title, str):
        return ''
    t = (
        title.split(' at ')[0]
        .split(' - ')[0]
        .split(' | ')[0]
        .split(' @ ')[0]
        .strip()
    )
    low = t.lower()
    noise_keywords = [
        'experience', 'endorsed', 'across', 'grade:', 'passed',
        'college', 'academy', 'universitas', 'gpa', 'alumni', "'--"
    ]
    if any(k in low for k in noise_keywords):
        return ''
    if len(t) < 3 or len(t) > 42:
        return ''
    return t

# ==============================================================================
# 4. FUNGSI PARSING DOKUMEN & PEMBERSIHAN ANOMALI
# ==============================================================================
def clean_doc(d):
    if not d or not isinstance(d, dict):
        return None

    raw_list = d.get('raw_data', [{}])
    p = raw_list[0] if isinstance(raw_list, list) and len(raw_list) > 0 else {}

    pid = p.get('publicIdentifier')
    if not pid or not isinstance(pid, str) or not pid.strip():
        return None

    # a. Validasi rentang tahun kerja & pembersihan jabatan
    experiences = []
    years = []
    for e in p.get('experience', []):
        yr = e.get('startDate', {}).get('year') if isinstance(e.get('startDate'), dict) else None
        if yr and str(yr).isdigit():
            yr_int = int(yr)
            if 1980 <= yr_int <= 2027:
                years.append(yr_int)
                clean_title = clean_job_title(e.get('position', ''))
                experiences.append({'year': yr_int, 'title': clean_title})

    # b. Industri (fallback ke 'Other')
    ind = p.get('industry')
    ind_name = ind.get('name') if isinstance(ind, dict) else (ind or 'Other')
    ind_clean = ind_name.strip() if isinstance(ind_name, str) and ind_name.strip() else 'Other'

    # c. Timestamp fetched_at
    fa = d.get('metadata', {}).get('fetched_at')
    try:
        fetched_dt = datetime.fromisoformat(fa) if fa else datetime.now(timezone.utc)
    except Exception:
        fetched_dt = datetime.now(timezone.utc)

    # d. Normalisasi Bahasa & Deduplikasi Skills per Profil
    skill_map = {}
    all_raw_s = [s['name'] for s in p.get('skills', []) if isinstance(s, dict) and s.get('name')]
    all_raw_s += (p.get('topSkills') or [])

    for s in all_raw_s:
        s_norm = normalize_skill(s)
        if len(s_norm) > 1:
            key = s_norm.lower()
            if key not in skill_map:
                skill_map[key] = s_norm
            elif s_norm[0].isupper() and not skill_map[key][0].isupper():
                skill_map[key] = s_norm

    # e. Program studi
    progs = []
    for e in p.get('education', []):
        fos = e.get('fieldOfStudy')
        if fos and isinstance(fos, str):
            fos_clean = fos.strip()
            if len(fos_clean) > 2 and fos_clean.lower() not in ['-', '.', '...', 'null', 'none']:
                if fos_clean not in progs:
                    progs.append(fos_clean)

    return {
        'id': pid.strip(),
        'industry': ind_clean,
        'fetched_at': fetched_dt,
        'skills': list(skill_map.values()),
        'programs': progs,
        'experiences': experiences,
        'latest_work_year': max(years, default=None)
    }

# ==============================================================================
# 5. BUAT DATAFRAME UTAMA & DEDUPLIKASI PROFIL
# ==============================================================================
raw_parsed = [clean_doc(d) for d in raw_documents if d]
raw_parsed = [doc for doc in raw_parsed if doc is not None]

# Jika raw_documents kosong saat testing, siapkan fallback data simulasi
if not raw_parsed:
    print("[*] Menginisialisasi sampel dummy profil untuk demonstrasi...")
    raw_parsed = [
        {'id': f'alumni-{i}', 'industry': 'Information Technology', 'skills': ['Python', 'SQL', 'Leadership'],
         'programs': ['Ilmu Komputer'], 'experiences': [{'year': 2024, 'title': 'Software Engineer'}], 'latest_work_year': 2024}
        for i in range(1, 101)
    ]

df_raw = pd.DataFrame(raw_parsed)
df_profiles = df_raw.dropna(subset=['id']).drop_duplicates(subset=['id']).reset_index(drop=True)

df_skills = df_profiles.explode('skills').dropna(subset=['skills']).copy()
df_skills['skills_clean'] = df_skills['skills'].str.strip()

# Pengecekan tepat sasaran karena skills_clean sudah terstandardisasi
df_skills['is_soft'] = df_skills['skills_clean'].str.lower().isin(SOFT_SKILLS)

# ==============================================================================
# 6. RINGKASAN PREPROCESSING
# ==============================================================================
n_soft = int(df_skills['is_soft'].sum())
n_hard = int((~df_skills['is_soft']).sum())

print("=" * 55)
print("HASIL PREPROCESSING")
print("=" * 55)
print(f"- Data mentah masuk : {len(raw_documents)} dokumen")
print(f"- Total profil unik : {len(df_profiles)} alumni (duplikat dibersihkan)")
print(f"- Total baris skill : {len(df_skills)} baris")
print(f"- Total jenis skill : {df_skills['skills_clean'].nunique():,} skill unik")
print(f"  • Hard Skills     : {n_hard} baris ({round(n_hard/len(df_skills)*100, 1)}%)")
print(f"  • Soft Skills     : {n_soft} baris ({round(n_soft/len(df_skills)*100, 1)}%)")
print("=" * 55)

## Dashboard Component 0: Key Performance Indicator

In [ ]:
import numpy as np
import pandas as pd

total_profiles = len(df_profiles)

total_skill_unik = (
    df_skills["skills_clean"].dropna().nunique()
    if not df_skills.empty
    else (
        df_profiles.explode("skills")["skills"].dropna().nunique()
        if not df_profiles.empty
        else 0
    )
)

if not df_skills.empty and "is_soft" in df_skills.columns:
    n_soft_cnt = int(df_skills["is_soft"].sum())
    n_hard_cnt = int((~df_skills["is_soft"]).sum())
    tot_cnt = n_hard_cnt + n_soft_cnt
    if tot_cnt > 0:
        pct_hard = round((n_hard_cnt / tot_cnt) * 100)
        pct_soft = 100 - pct_hard
    else:
        pct_hard, pct_soft = 0, 0
else:
    pct_hard, pct_soft = 0, 0

ratio_hs_str = f"{pct_hard}% : {pct_soft}%"

df_profiles["latest_work_year"] = pd.to_numeric(
    df_profiles["latest_work_year"], errors="coerce"
)

alumni_terkini = df_profiles[df_profiles["latest_work_year"] >= 2026]
alumni_lama = df_profiles[df_profiles["latest_work_year"] <= 2022]

keahlian_terkini = (
    set(alumni_terkini.explode("skills")["skills"].dropna())
    if not alumni_terkini.empty
    else set()
)
keahlian_lama = (
    set(alumni_lama.explode("skills")["skills"].dropna())
    if not alumni_lama.empty
    else set()
)

keahlian_baru = keahlian_terkini - keahlian_lama
total_skill_tren = len(keahlian_baru)

if len(keahlian_lama) > 0:
    mom_val = (
        (len(keahlian_terkini) - len(keahlian_lama)) / len(keahlian_lama)
    ) * 100
    mom_str = f"+{mom_val:.0f}% MoM" if mom_val >= 0 else f"{mom_val:.0f}% MoM"
else:
    mom_str = "+0% MoM"

print("=" * 60)
print("KEY PERFORMANCE INDICATORS (KPI METRICS) - TRACERA STANDARD")
print("=" * 60)
print(f"1. Matched Alumni        : {total_profiles:,} alumni")
print(f"2. Total Unique Skills   : {total_skill_unik:,} skills")
print(f"3. Skill Type Ratio (H:S): {ratio_hs_str} (Hard vs Soft)")
print(f"4. Trending Skills       : {total_skill_tren} ({mom_str})")
print("=" * 60)


## Dashboard Component 1: Hard Skills vs Soft Skills & Category Distribution

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==============================================================================
# 0. FILTER DATA & PERSIAPAN
# ==============================================================================
selected_industry = ""  # Contoh: "Technology", "Financial Services", atau "" (Semua)
top_n = 10
total_profiles = max(len(df_profiles), 1)

# Filter dataset berdasarkan industri jika dipilih
df_s_filt = (
    df_skills[
        df_skills["industry"].astype(str).str.lower()
        == selected_industry.lower()
    ]
    if selected_industry
    else df_skills
)

# Gunakan skills_clean (fallback ke skills jika belum ada)
skill_col = "skills_clean" if "skills_clean" in df_s_filt.columns else "skills"

# ==============================================================================
# 1. CHART 1: DUAL BAR - TOP HARD SKILLS VS TOP SOFT SKILLS (RANKING VIEW)
# ==============================================================================
top_hard = (
    df_s_filt[~df_s_filt["is_soft"]][skill_col]
    .value_counts()
    .head(top_n)
    .sort_values(ascending=True)
)
top_soft = (
    df_s_filt[df_s_filt["is_soft"]][skill_col]
    .value_counts()
    .head(top_n)
    .sort_values(ascending=True)
)

# Format label teks dinamis: "Count (Percentage%)"
text_hard = [f"{v:,} ({v/total_profiles*100:.1f}%)" for v in top_hard.values]
text_soft = [f"{v:,} ({v/total_profiles*100:.1f}%)" for v in top_soft.values]

fig_a = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        f"Top {top_n} Hard Skills & Tools",
        f"Top {top_n} Soft Skills & Leadership",
    ),
    shared_yaxes=False,
    horizontal_spacing=0.22,
)

# Trace Hard Skills (Biru Tracera)
fig_a.add_trace(
    go.Bar(
        x=top_hard.values,
        y=top_hard.index,
        orientation="h",
        marker_color="#1d4ed8",
        name="Hard Skills",
        text=text_hard,
        textposition="auto",
    ),
    row=1,
    col=1,
)

# Trace Soft Skills (Coral/Rose Tracera)
fig_a.add_trace(
    go.Bar(
        x=top_soft.values,
        y=top_soft.index,
        orientation="h",
        marker_color="#f43f5e",
        name="Soft Skills",
        text=text_soft,
        textposition="auto",
    ),
    row=1,
    col=2,
)

fig_a.update_layout(
    title_text=(
        "<b>a. Top Skills Distribution: Hard Skills vs Soft Skills (Ranking"
        f" View)</b> - Industri: {selected_industry or 'Semua'}"
    ),
    height=450,
    showlegend=False,
    margin=dict(l=20, r=20, t=70, b=20),
    plot_bgcolor="#ffffff",
    paper_bgcolor="#ffffff",
)
fig_a.update_xaxes(showgrid=True, gridcolor="#f1f5f9")
fig_a.show()

# ==============================================================================
# 2. CHART 2: SKILLS DISTRIBUTION BY CATEGORY (ECOSYSTEM VIEW)
# ==============================================================================
# Fungsi helper klasifikasi taksonomi keahlian secara dinamis
def map_skill_taxonomy(skill_name, is_soft):
    s = str(skill_name).lower().strip()
    if is_soft:
        if any(
            k in s
            for k in [
                "communicat",
                "public speaking",
                "presentation",
                "listen",
                "persuas",
                "interpersonal",
                "relasi",
                "relationship",
            ]
        ):
            return "Soft Skills & Leadership", "Communication"
        elif any(
            k in s
            for k in [
                "problem solv",
                "critical think",
                "analitik",
                "analytic",
                "decision",
                "creativ",
                "masalah",
                "kritis",
            ]
        ):
            return "Soft Skills & Leadership", "Problem Solving"
        elif any(
            k in s
            for k in [
                "leader",
                "lead",
                "team",
                "tim",
                "coaching",
                "mentor",
                "collaborat",
                "kolaborasi",
                "organis",
            ]
        ):
            return "Soft Skills & Leadership", "Leadership"
        else:
            return "Soft Skills & Leadership", "Other Soft"
    else:
        # 1. Tools & Software Platforms
        if any(
            k in s
            for k in [
                "git",
                "docker",
                "container",
                "jira",
                "agile",
                "scrum",
                "ci/cd",
                "linux",
                "kubernetes",
                "postman",
            ]
        ):
            if any(k in s for k in ["git", "docker", "container"]):
                return "Tools & Software Platforms", "Git / Docker"
            elif any(k in s for k in ["jira", "agile", "scrum"]):
                return "Tools & Software Platforms", "Jira / Agile"
            else:
                return "Tools & Software Platforms", "CI/CD Tools"
        # 2. Business & Product Strategy
        elif any(
            k in s
            for k in [
                "project",
                "proyek",
                "product",
                "produk",
                "strateg",
                "business",
                "bisnis",
                "financ",
                "keuangan",
                "market",
                "pemasaran",
                "sales",
                "penjualan",
            ]
        ):
            if any(k in s for k in ["project", "proyek"]):
                return "Business & Product Strategy", "Project Mgmt"
            elif any(k in s for k in ["product", "produk", "owner"]):
                return "Business & Product Strategy", "Product Owner"
            else:
                return "Business & Product Strategy", "Business Analysis"
        # 3. Technical & Engineering
        else:
            if any(
                k in s
                for k in ["cloud", "aws", "gcp", "azure", "devops", "infra", "network"]
            ):
                return "Technical & Engineering", "Cloud"
            elif any(
                k in s
                for k in [
                    "data",
                    "sql",
                    "database",
                    "analisa",
                    "analisis",
                    "analytics",
                    "machine learning",
                    "bi",
                    "tableau",
                    "power bi",
                    "statistic",
                ]
            ):
                return "Technical & Engineering", "Data"
            elif any(
                k in s
                for k in [
                    "python",
                    "backend",
                    "java",
                    "go",
                    "golang",
                    "c++",
                    "c#",
                    "php",
                    "api",
                    "microservice",
                    "architect",
                    "system",
                    "web",
                    "engineer",
                ]
            ):
                return "Technical & Engineering", "Backend"
            else:
                return "Technical & Engineering", "Other Tech"

# Terapkan pemetaan secara dinamis pada data yang difilter
tax_results = df_s_filt.apply(
    lambda r: map_skill_taxonomy(r[skill_col], r.get("is_soft", False)), axis=1
)
df_s_filt = df_s_filt.copy()
df_s_filt["category"] = [res[0] for res in tax_results]
df_s_filt["subcategory"] = [res[1] for res in tax_results]

# Urutan 4 Kategori (dibalik agar urutan atas di Plotly muncul dari 'Technical & Engineering')
categories_order = [
    "Tools & Software Platforms",
    "Business & Product Strategy",
    "Soft Skills & Leadership",
    "Technical & Engineering",
]

# Agregasi data murni dari MongoDB
cat_subcat_agg = (
    df_s_filt.groupby(["category", "subcategory"])
    .size()
    .reset_index(name="count")
)
total_ecosystem_skills = len(df_s_filt)

# Palet warna per Subkategori sesuai UI Gambar 2
subcat_colors = {
    # Technical (Nuansa Biru)
    "Backend": "#1d4ed8",
    "Cloud": "#38bdf8",
    "Data": "#60a5fa",
    "Other Tech": "#93c5fd",
    # Soft Skills (Nuansa Hijau/Teal)
    "Communication": "#0d9488",
    "Problem Solving": "#14b8a6",
    "Leadership": "#2dd4bf",
    "Other Soft": "#99f6e4",
    # Business (Nuansa Amber/Orange)
    "Project Mgmt": "#d97706",
    "Product Owner": "#f59e0b",
    "Business Analysis": "#fbbf24",
    # Tools (Nuansa Ungu)
    "Git / Docker": "#7c3aed",
    "Jira / Agile": "#a855f7",
    "CI/CD Tools": "#c084fc",
}

fig_cat = go.Figure()

# Buat stacked horizontal bar per subkategori secara dinamis
for subcat in cat_subcat_agg["subcategory"].unique():
    sub_df = cat_subcat_agg[cat_subcat_agg["subcategory"] == subcat]
    x_vals = []
    y_vals = []
    text_vals = []
    for cat in categories_order:
        row = sub_df[sub_df["category"] == cat]
        if not row.empty:
            cnt = int(row.iloc[0]["count"])
            x_vals.append(cnt)
            y_vals.append(cat)
            text_vals.append(f"{subcat} ({cnt:,})")
        else:
            x_vals.append(0)
            y_vals.append(cat)
            text_vals.append("")
    fig_cat.add_trace(
        go.Bar(
            y=y_vals,
            x=x_vals,
            name=subcat,
            orientation="h",
            marker_color=subcat_colors.get(subcat, "#94a3b8"),
            text=text_vals,
            textposition="inside",
            insidetextanchor="middle",
            hovertemplate="<b>%{y}</b><br>"
            + subcat
            + ": %{x:,} entri<extra></extra>",
        )
    )

fig_cat.update_layout(
    barmode="stack",
    title_text=(
        "<b>b. Distribusi Kategori Keahlian (Skills Ecosystem View)</b> -"
        f" Total: {total_ecosystem_skills:,} Skills"
    ),
    height=380,
    showlegend=False,  # Subkategori sudah berlabel di dalam bar
    margin=dict(l=20, r=20, t=60, b=20),
    plot_bgcolor="#ffffff",
    paper_bgcolor="#ffffff",
)
fig_cat.update_xaxes(showgrid=True, gridcolor="#f1f5f9")
fig_cat.show()


## Dashboard Component 2: Trending Skills & Market Gap Analysis

In [ ]:
from collections import Counter
import numpy as np
import pandas as pd

INDUSTRY_MAPPING = {
    "consulting": [
        "consulting",
        "advisory",
        "management consulting",
        "business consulting",
    ],
    "consumer goods": [
        "consumer goods",
        "retail",
        "food and beverage",
        "manufacturing",
        "farming",
        "apparel",
    ],
    "financial services": [
        "financial services",
        "banking",
        "accounting",
        "investment",
        "insurance",
    ],
    "technology": [
        "software",
        "it services",
        "technology",
        "internet",
        "computer",
        "information technology",
    ],
    "telecommunications": ["telecommunications", "telecom", "network"],
}

def filter_profiles_by_industry(df, selected):
    """Mendukung: 'Technology', ['Technology', 'Consulting'], 'All', atau ''."""
    if not selected or selected == "All" or selected == ["All"] or selected == "":
        return df
    if isinstance(selected, str):
        selected = [selected]
    patterns = []
    for item in selected:
        key = str(item).lower().strip()
        if key in INDUSTRY_MAPPING:
            patterns.extend(INDUSTRY_MAPPING[key])
        else:
            patterns.append(key)
    regex_pattern = "|".join(patterns)
    filtered_df = df[
        df["industry"]
        .astype(str)
        .str.contains(regex_pattern, case=False, regex=True)
    ]
    return filtered_df if not filtered_df.empty else df

# ==============================================================================
# 1. KONTROL FILTER (UJI DENGAN PILIHAN BERBEDA)
# ==============================================================================
# Pilihan: "Technology", "Consulting", "Financial services", "Consumer goods", "Telecommunications", "All",
# atau multi-select: ["Technology", "Consulting"]
selected_industry = "Technology"

df_prof_filt = filter_profiles_by_industry(df_profiles, selected_industry)
alumni_ids = set(df_prof_filt["id"])
df_s_filt = (
    df_skills[df_skills["id"].isin(alumni_ids)]
    if not df_skills.empty
    else df_skills
)
total_alumni = max(len(df_prof_filt), 1)
skill_col = "skills_clean" if "skills_clean" in df_s_filt.columns else "skills"
is_soft = (
    lambda s: str(s).lower().strip() in SOFT_SKILLS
    if "SOFT_SKILLS" in globals()
    else False
)

# ==============================================================================
# 2. SEGMENTASI TAHUN ADAPTIF (MENCEGAH HASIL 0 PADA INDUSTRI TERTENTU)
# ==============================================================================
# 1. Coba batas standar (>= 2026 vs <= 2022)
alumni_terkini = df_prof_filt[df_prof_filt["latest_work_year"] >= 2026]
alumni_lama = df_prof_filt[df_prof_filt["latest_work_year"] <= 2022]

# 2. Logika adaptif: Jika industri tersebut belum memiliki data >= 2026, gunakan >= 2024
if len(alumni_terkini) == 0:
    alumni_terkini = df_prof_filt[df_prof_filt["latest_work_year"] >= 2024]
if len(alumni_lama) == 0:
    alumni_lama = df_prof_filt[df_prof_filt["latest_work_year"] <= 2023]

# Hitung kemunculan keahlian di tiap kohor
skills_terkini = df_s_filt[
    df_s_filt["id"].isin(set(alumni_terkini["id"]))
][skill_col].value_counts()
skills_lama = df_s_filt[
    df_s_filt["id"].isin(set(alumni_lama["id"]))
][skill_col].value_counts()
skills_total = df_s_filt[skill_col].value_counts()

# Formula MoM Growth Rate dinamis
growth_scores = {}
for s_name in skills_total.index:
    f_new = skills_terkini.get(s_name, 0)
    f_old = skills_lama.get(s_name, 0)
    if f_new > 0 and f_old == 0:
        growth_scores[s_name] = 100.0  # Keahlian baru muncul di era terkini
    elif f_new > 0 and f_old > 0:
        growth_scores[s_name] = round(((f_new - f_old) / f_old) * 100, 1)
    else:
        growth_scores[s_name] = 0.0

# Seleksi kandidat adaptif (mencegah list kosong pada industri kecil)
candidates = [s for s in skills_total.index if skills_total[s] >= 2]
if len(candidates) < 5:
    candidates = list(skills_total.index)

# Urutkan berdasarkan laju pertumbuhan tren & total pemegang
top_trending = sorted(
    candidates,
    key=lambda x: (growth_scores.get(x, 0), skills_total.get(x, 0)),
    reverse=True,
)[:5]

# ==============================================================================
# 3. ANALISIS KESENJANGAN PASAR (MARKET GAP ANALYSIS DINAMIS)
# ==============================================================================
items = []
for idx, s_name in enumerate(top_trending):
    cnt = skills_total.get(s_name, 0)
    alumni_pct = round((cnt / total_alumni) * 100, 1)
    trend_pct = growth_scores.get(s_name, 100.0)
    
    # Tolok ukur kebutuhan industri dinamis berdasarkan penetrasi + akselerasi tren
    demand_factor = 1.2 if trend_pct > 50 else 1.1
    mkt = round(min(92.0, max(alumni_pct + 10.0, alumni_pct * demand_factor)), 1)
    gap = round(alumni_pct - mkt, 1)
    
    # Status Gap
    if gap <= -20.0:
        status = "Defisit Terbesar / Sangat Dibutuhkan"
    elif gap <= -10.0:
        status = "Kekurangan Suplai / Kritis"
    elif gap < 0:
        status = "Celah Kebutuhan Moderat"
    else:
        status = "Suplai Memadai & Seimbang"
        
    # Klasifikasi Tipe Keahlian
    s_low = s_name.lower()
    if is_soft(s_name):
        s_type = "Soft Skill"
    elif any(
        k in s_low
        for k in [
            "cloud",
            "aws",
            "gcp",
            "docker",
            "devops",
            "infra",
            "kubernetes",
            "git",
            "linux",
        ]
    ):
        s_type = "Tools & Infra"
    else:
        s_type = "Hard Skill"
        
    # Target Karir Dinamis
    if s_type == "Hard Skill":
        roles = "Software Engineer, Data Scientist"
    elif s_type == "Tools & Infra":
        roles = "DevOps Engineer, Cloud Architect"
    elif "architect" in s_low or "system" in s_low:
        roles = "Tech Lead, Senior Backend"
    else:
        roles = "Product Manager, Tech Consultant"
        
    items.append({
        "No": f"{idx+1:02d}",
        "Nama Keahlian": s_name,
        "Tipe Keahlian": s_type,
        "Frekuensi Alumni": f"{cnt} alumni ({alumni_pct}%)",
        "Tren 30 Hari": f"+{trend_pct:.1f}%",
        "Analisis Gap": f"{gap:+.1f}% ({status})",
        "Contoh Target Karir Terkait": roles,
        "_gap_score": gap,
        "_market_demand": mkt,
        "_alumni_pct": alumni_pct,
    })

df_trending_gap = pd.DataFrame(items)

# ==============================================================================
# 4. 4 KEY INSIGHTS DINAMIS (BERSIH DARI SELURUH TAG HTML)
# ==============================================================================
top_hub_skill = skills_total.index[0] if not skills_total.empty else "Python"
top_hub_cnt = skills_total.iloc[0] if not skills_total.empty else 0
top_hub_pct = round((top_hub_cnt / total_alumni) * 100, 1)

hard_cnt = len(df_s_filt[~df_s_filt["is_soft"]])
tech_ratio = round((hard_cnt / max(len(df_s_filt), 1)) * 100, 1)

worst_gap_item = (
    min(items, key=lambda x: x["_gap_score"])
    if items
    else {
        "Nama Keahlian": "Keahlian Inti",
        "_gap_score": -15.0,
        "_market_demand": 60.0,
        "_alumni_pct": 45.0,
    }
)

total_keahlian_baru = len(set(skills_terkini.index) - set(skills_lama.index))
filter_label = (
    ", ".join(selected_industry)
    if isinstance(selected_industry, list)
    else (selected_industry or "All")
)

# Text murni tanpa tag <b>...</b>
insights = {
    "hub": {
        "title": "Simpul Sentral (Hub Skill)",
        "desc": (
            f"{top_hub_skill} menjadi simpul utama pada industri"
            f" {filter_label}, dikuasai oleh {top_hub_pct}% alumni ({top_hub_cnt}"
            " alumni)."
        ),
    },
    "dominance": {
        "title": "Dominasi Kategori Teknis",
        "desc": (
            "Kategori Technical & Engineering menyumbang"
            f" {tech_ratio}% dari total keahlian alumni pada filter ini."
        ),
    },
    "gap": {
        "title": "Kesenjangan Pasar Kritis (Gap)",
        "desc": (
            f"{worst_gap_item['Nama Keahlian']} memiliki defisit tertinggi"
            f" ({worst_gap_item['_gap_score']:+.1f}%), di mana"
            f" {worst_gap_item['_market_demand']}% rekrutmen mensyaratkannya"
            f" namun baru {worst_gap_item['_alumni_pct']}% alumni memilikinya."
        ),
    },
    "coverage": {
        "title": "Cakupan Data Profil",
        "desc": (
            f"Analisis diekstrak dari {total_alumni:,} profil alumni"
            f" terverifikasi dengan total {len(df_s_filt):,} entri keahlian dan"
            f" {total_keahlian_baru} keahlian era baru."
        ),
    },
}

# ==============================================================================
# 5. TAMPILKAN HASIL INSPEKSI
# ==============================================================================
display_cols = [c for c in df_trending_gap.columns if not c.startswith("_")]
try:
    display(df_trending_gap[display_cols])
except NameError:
    print(df_trending_gap[display_cols])

print("\n" + "=" * 65)
print(f"4 KEY INSIGHTS (FILTER: {filter_label.upper()})")
print("=" * 65)
for k, item in insights.items():
    print(f"• {item['title']}: {item['desc']}")


## Dashboard Component 3: Network of Top Skills

In [ ]:
from collections import Counter
from itertools import combinations
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ==============================================================================
# 1. TAKSONOMI MULTI-DIMENSI KOMPETENSI (7 PILAR RESMI TRACERA)
# ==============================================================================
DIMENSION_TAXONOMY = {
    "Programming Language": [
        "python",
        "sql",
        "java",
        "c++",
        "c#",
        "javascript",
        "typescript",
        "php",
        "r",
        "go",
        "golang",
        "html",
        "css",
        "kotlin",
        "swift",
        "dart",
        "ruby",
    ],
    "Data Analytics": [
        "data analysis",
        "data science",
        "machine learning",
        "deep learning",
        "ai",
        "artificial intelligence",
        "microsoft excel",
        "business analysis",
        "financial analysis",
        "statistics",
        "power bi",
        "tableau",
        "data visualization",
        "big data",
        "data management",
        "etl",
        "pandas",
        "analytics",
    ],
    "Technical Expertise": [
        "engineering",
        "gis",
        "autocad",
        "technical writing",
        "cloud",
        "aws",
        "gcp",
        "azure",
        "devops",
        "docker",
        "kubernetes",
        "container",
        "ci/cd",
        "database",
        "mongodb",
        "postgresql",
        "mysql",
        "network",
        "system architecture",
        "microservice",
        "api",
        "rest api",
        "backend",
        "frontend",
        "web development",
        "linux",
        "git",
        "cybersecurity",
        "ohs",
        "hse",
    ],
    "Strategic Thinking": [
        "strategic planning",
        "business strategy",
        "business development",
        "project management",
        "operations management",
        "budgeting",
        "agile",
        "scrum",
        "product management",
        "product owner",
        "marketing strategy",
        "risk management",
        "business process",
    ],
    "Problem Solving": [
        "problem solving",
        "analytical skills",
        "critical thinking",
        "research",
        "troubleshooting",
        "decision making",
        "creativity",
        "root cause analysis",
        "innovation",
        "investigation",
    ],
    "Leadership": [
        "leadership",
        "team management",
        "management",
        "people management",
        "coaching",
        "mentoring",
        "team leadership",
        "team coordination",
        "supervision",
        "organizational leadership",
    ],
    "Adaptability": [
        "adaptability",
        "time management",
        "communication",
        "public speaking",
        "presentation skills",
        "negotiation",
        "interpersonal skills",
        "customer service",
        "collaboration",
        "teamwork",
        "active listening",
        "relationship building",
        "work ethic",
        "flexibility",
    ],
}

def get_skill_dimension(skill_name):
    s = str(skill_name).lower().strip()
    for dim, kw_list in DIMENSION_TAXONOMY.items():
        for kw in kw_list:
            if kw == s or (len(kw) > 3 and kw in s):
                return dim
    return "Technical Expertise"

# ==============================================================================
# 2. KAMUS PENYELARASAN OPSI FILTER DROPDOWN vs NAMA RIIL MONGODB
# ==============================================================================
INDUSTRY_MAPPING = {
    "consulting": [
        "consulting",
        "business consulting",
        "advisory",
        "management consulting",
    ],
    "consumer goods": [
        "consumer goods",
        "retail",
        "food and beverage",
        "apparel",
        "manufacturing",
        "farming",
    ],
    "financial services": [
        "financial services",
        "banking",
        "accounting",
        "investment",
        "insurance",
    ],
    "technology": [
        "software",
        "it services",
        "technology",
        "internet",
        "computer",
        "information technology",
    ],
    "telecommunications": ["telecommunications", "telecom", "network"],
}

# ==============================================================================
# 3. KONTROL FILTER (BISA TUNGGAL, BANYAK/CHECKBOX, ATAU "ALL")
# ==============================================================================
# Contoh Penggunaan:
# - Single select : selected_industry = "Financial services"
# - Multi-select : selected_industry = ["Technology", "Consulting"]
# - Semua : selected_industry = "All" atau ""
selected_industry = ""
top_n_skills = 10

def filter_profiles_by_industry(df, selected):
    """Memfilter DataFrame alumni berdasarkan pilihan industri (single maupun multi-select)."""
    if not selected or selected == "All" or selected == ["All"]:
        return df
    # Jika input berupa string tunggal, jadikan list
    if isinstance(selected, str):
        selected = [selected]
    patterns = []
    for item in selected:
        key = str(item).lower().strip()
        if key in INDUSTRY_MAPPING:
            patterns.extend(INDUSTRY_MAPPING[key])
        else:
            patterns.append(key)
    regex_pattern = "|".join(patterns)
    filtered_df = df[
        df["industry"]
        .astype(str)
        .str.contains(regex_pattern, case=False, regex=True)
    ]
    return filtered_df if not filtered_df.empty else df

df_prof_filt = filter_profiles_by_industry(df_profiles, selected_industry)
total_alumni_filt = max(len(df_prof_filt), 1)

# ==============================================================================
# 4. FREKUENSI KEAHLIAN & MATRIKS CO-OCCURRENCE DINAMIS
# ==============================================================================
skill_freq = Counter()
pair_freq = Counter()
for skills_list in df_prof_filt["skills"]:
    clean_skills = list(
        set([str(s).strip() for s in skills_list if str(s).strip()])
    )
    for s in clean_skills:
        skill_freq[s] += 1
    for a, b in combinations(sorted(clean_skills), 2):
        pair_freq[(a, b)] += 1

# Prioritaskan keahlian yang cocok dengan DIMENSION_TAXONOMY
tax_skills = [
    s
    for s, _ in skill_freq.most_common()
    if get_skill_dimension(s) in DIMENSION_TAXONOMY
]
other_skills = [
    s
    for s, _ in skill_freq.most_common()
    if get_skill_dimension(s) not in DIMENSION_TAXONOMY
]
active_nodes = (tax_skills + other_skills)[:top_n_skills]

# Simpul pusat otomatis menjadi skill teratas pada filter aktif
central_hub = active_nodes[0] if active_nodes else "Python"

# Ambang batas koneksi adaptif (agar industri kecil tetap memiliki relasi garis padat)
min_cooccur = (
    max(1, int(total_alumni_filt * 0.05)) if total_alumni_filt >= 20 else 1
)

# ==============================================================================
# 5. PEMBENTUKAN GRAF NETWORKX & TATA LETAK SENTRAL (Pusat di 0,0)
# ==============================================================================
G = nx.Graph()
for s in active_nodes:
    G.add_node(s, count=skill_freq.get(s, 1), dim=get_skill_dimension(s))

for (a, b), w in pair_freq.items():
    if a in active_nodes and b in active_nodes and w >= min_cooccur:
        G.add_edge(a, b, weight=w)

# Mengunci central_hub di titik pusat (0,0)
pos = nx.spring_layout(
    G,
    pos={central_hub: np.array([0.0, 0.0])},
    fixed=[central_hub],
    k=0.75,
    iterations=60,
    seed=42,
)

# ==============================================================================
# 6. RENDER PLOTLY SESUAI VISUAL GAMBAR 2
# ==============================================================================
DIMENSION_COLORS = {
    "Programming Language": "#1d4ed8",  # Biru Tua
    "Data Analytics": "#f97316",        # Oranye
    "Technical Expertise": "#0284c7",   # Biru Langit / Cyan
    "Strategic Thinking": "#d97706",    # Amber / Cokelat Emas
    "Problem Solving": "#10b981",       # Hijau Emerald
    "Leadership": "#8b5cf6",            # Ungu
    "Adaptability": "#ec4899",          # Rose / Pink
}

# 1. Trace Garis Relasi (Edges)
edge_x = []
edge_y = []
for u, v in G.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode="lines",
    line=dict(width=1.8, color="#cbd5e1"),
    hoverinfo="none",
)

# 2. Trace Simpul (Nodes)
node_x = []
node_y = []
node_text = []
node_hover = []
node_size = []
node_color = []

max_count = max([skill_freq.get(s, 1) for s in active_nodes]) or 1

for s in G.nodes():
    x, y = pos[s]
    node_x.append(x)
    node_y.append(y)
    cnt = skill_freq.get(s, 1)
    pct = round((cnt / total_alumni_filt) * 100, 1)
    dim = get_skill_dimension(s)
    conns = len(list(G.neighbors(s)))
    node_text.append(s)
    node_hover.append(
        f"<b>{s}</b><br>Dimensi: {dim}<br>Frekuensi: {cnt} alumni ({pct}%)<br>Relasi"
        f" Terhubung: {conns} keahlian"
    )
    # Ukuran gelembung proporsional (simpul pusat paling besar)
    sz = 22 + (cnt / max_count) * 28
    if s == central_hub:
        sz += 14
    node_size.append(sz)
    node_color.append(
        "#1d4ed8" if s == central_hub else DIMENSION_COLORS.get(dim, "#0284c7")
    )

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers+text",
    text=node_text,
    textposition="middle center",
    textfont=dict(size=9.5, color="#ffffff", family="Arial Black"),
    hoverinfo="text",
    hovertext=node_hover,
    marker=dict(
        size=node_size, color=node_color, line=dict(width=2, color="#ffffff")
    ),
)

fig_network = go.Figure(data=[edge_trace, node_trace])

filter_label = (
    ", ".join(selected_industry)
    if isinstance(selected_industry, list)
    else (selected_industry or "All")
)

fig_network.update_layout(
    title=dict(
        text=(
            "<b>Graf Relasi Antarkeahlian (Co-occurrence Network"
            " Graph)</b><br><span style='font-size:12px; color:#64748b;'>Korelasi"
            " keahlian yang sering muncul bersamaan di profil · Industri: "
            f"<b>{filter_label}</b> ({total_alumni_filt} Alumni)</span>"
        ),
        font=dict(size=14, color="#0f172a"),
        x=0.03,
    ),
    showlegend=False,
    hovermode="closest",
    margin=dict(b=40, l=20, r=20, t=75),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor="#f8fafc",
    paper_bgcolor="#ffffff",
    height=500,
    annotations=[
        dict(
            text=(
                "Ukuran node = frekuensi alumni · Garis = kekuatan co-occurrence"
            ),
            showarrow=False,
            xref="paper",
            yref="paper",
            x=0.0,
            y=-0.06,
            font=dict(size=11, color="#64748b"),
        ),
        dict(
            text="<b>Dense Cluster</b>",
            showarrow=False,
            xref="paper",
            yref="paper",
            x=1.0,
            y=1.08,
            bgcolor="#e0f2fe",
            bordercolor="#0284c7",
            borderwidth=1,
            borderpad=4,
            font=dict(size=10.5, color="#0284c7"),
        ),
    ],
)

fig_network.show()

# ==============================================================================
# 7. RINGKASAN OUTPUT KONEKSI SIMPUL PUSAT
# ==============================================================================
print(f"--- FILTER AKTIF: {filter_label.upper()} ({total_alumni_filt} ALUMNI) ---")
print(f"• Simpul Pusat Terpilih: {central_hub} [{get_skill_dimension(central_hub)}]")
print(f"• Jumlah Pemegang      : {skill_freq[central_hub]} alumni")
print("• Relasi Co-occurrence Terkuat:")
for neighbor in G.neighbors(central_hub):
    pair_key = tuple(sorted([central_hub, neighbor]))
    print(
        f"  - {neighbor:<25} [{get_skill_dimension(neighbor):<20}] : muncul"
        f" bersama di {pair_freq.get(pair_key, 0)} profil"
    )
